# AI Driving Scene Analyzer - Notebook de référence

> Etat actuel: pipeline fonctionnel avant fine-tuning.

Ce notebook documente la version opérationnelle du projet:
- Détection avec YOLOv8 pré-entraîné
- Analyse de scène avec LLM (résumé, risque, recommandations)
- Interface Streamlit pour tester image par image

Pipeline:
`Image -> CV (YOLO) -> JSON détections -> LLM -> rapport + audio`

## Setup et exécution (état actuel)

### 1) Environnement Python
Utiliser le venv local du projet (`.venv`).

### 2) Dépendances
Installer avec `requirements.txt` dans le venv.

### 3) Variables d'environnement
Créer un fichier `.env` à la racine avec:
- `API_KEY=<cle_groq>`

### 4) Lancement Streamlit (obligatoire avec le bon interpréteur)
Commande recommandée:
- `.venv/Scripts/python.exe -m streamlit run app/app.py`

### 5) Réglages CV recommandés (pré-fine-tuning)
- conf: 0.40
- iou: 0.50
- raw YOLO: False
- classes CSV: `car,truck,bus,person,bicycle,motorcycle,traffic light,stop sign,train`
- filtre dashcam: True
- mode nuit auto: True

### 6) Limites actuelles
- Pas de fine-tuning: qualité variable selon éclairage et angle dashcam
- Fausse détection possible sur scènes nocturnes/fort reflet
- Variabilité entre images normale à ce stade

In [ ]:
from pathlib import Path
from cv.detector import YOLODetector

# Sanity check CV sans passer par l'UI
detector = YOLODetector("yolov8n")
print("YOLO chargé:", detector.model is not None)

images = sorted(Path("data/test").glob("*.jpg"))
if not images:
    print("Aucune image trouvée dans data/test")
else:
    img = images[0]
    dets = detector.detect(
        str(img),
        conf=0.4,
        iou=0.5,
        classes=[
            "car", "truck", "bus", "person", "bicycle",
            "motorcycle", "traffic light", "stop sign", "train",
        ],
        raw_output=False,
    )
    print("Image:", img)
    print("Nb detections:", len(dets))
    for d in dets[:10]:
        print(f"- {d.class_name} {d.confidence:.2f} {tuple(round(v, 1) for v in d.bbox)}")

## Commandes utiles

> Lancer l'application
- `.venv/Scripts/python.exe -m streamlit run app/app.py`

 > Tester CV seul
- `.venv/Scripts/python.exe scripts/test_yolo_on_data.py --data data/test --model yolov8n --conf 0.4 --iou 0.5 --classes "car,truck,bus,person,bicycle,motorcycle,traffic light,stop sign,train" --max 10 --top 10`

 > Générer les détections JSON
- `.venv/Scripts/python.exe scripts/test_yolo_on_data.py --data data/test --model yolov8n --max 50 --out outputs/detections.json`

 > Validation des détections
- `.venv/Scripts/python.exe scripts/validate_detections.py --input outputs/detections.json --classes car,person,truck,traffic\ light`

## Roadmap
- Stabiliser la qualité CV sur scènes complexes
- Fine-tuning YOLO sur BDD100K
- Mesurer les gains (mAP, precision, recall) et comparer au pré-entraîné